# Parameter sensitivity analysis

[The cluster selection strategies notebook](./demo_selection_strategies.ipynb)
demonstrates PLSCAN is less sensitive to the ``min_samples`` parameter ($k$)
than HDBSCAN*. This notebook runs a more comprehensive parameter sensitivity
analysis to determine whether that pattern holds on other datasets. This
parameter sensitivity analysis tells us how the clustering quality changes as a
result of changing $k$. The resulting value is independent of $k$ itself.
Instead they relate to a change in $k$: $\Delta k$!

We apply the analysis [Peng et al.
2022](https://www.nature.com/articles/s41467-022-33136-9#Sec18) used to evaluate
their clustering algorithm. They modified a "Latin-Hypercube
One-factor-At-a-Time (LH-OAT)" analysis. This sounds more complicated than it
is, especially for algorithms with a single parameter. The steps are as follows:

- Choose a parameter $k$ and divided its value-space in regularly sized
  segments.
- Choose a constant perturbation $\Delta v$ to evaluate.
- Sample values $v$ in each segment. We denote the set of sampled values with
  $V$. 
- Perturb the sampled values with $\pm \Delta v$, randomly choosing the positive
  or negative direction.
- Compute the algorithm's clustering quality scores at $v$ and $v \pm \Delta v$.
  For example, using the
  [ARI](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html).
  We denote this score as $ARI(k=v)$, indicating a score at parameter value
  $k=v$.
- Compute the sensitivity at $\Delta v$ using: 
$$
  s_{\Delta v} = \frac{1}{|V|} \sum_{v \in V} \left|\frac{ARI(k=v \pm \Delta v) - ARI(k=v)} {ARI(k=v \pm\Delta v) + ARI(k=v)}\right|. 
$$

We want to evaluate the sensitivity for multiple perturbation sizes $\Delta v$,
algorithm configurations, and datasets. Our implementation splits the work in
three stages and considers several clustering quality measures:

1. Sample the parameter and perturbations, collecting all parameter values to
   evaluate.
2. Compute the algorithms' clustering quality measures values on all datasets
   with the collected parameter values. Store these values!
3. Use the computed quality measures values to compute average sensitivities for
   each perturbation size.

In [ ]:
import warnings

import os
import time
import numpy as np
import pandas as pd
from tqdm import tqdm
from itertools import product
from scipy.stats import rankdata
from collections import defaultdict
from sklearn.decomposition import PCA
from sklearn.datasets import fetch_openml, load_iris as sk_load_iris
from sklearn.preprocessing import StandardScaler, normalize

from hdbscan import HDBSCAN
from fast_plscan import PLSCAN
from sklearn.metrics import adjusted_rand_score, homogeneity_score, completeness_score

from lib.plotting import sns, plt, mpl, lighten, frame_off
from lib.drawing import regplot_lowess_ci


plt.rcParams["figure.dpi"] = 150
plt.rcParams["figure.figsize"] = (2.75, 0.618 * 2.75)

## 1. Sample parameter values and perturbations

This cell performs the parameter value sampling and applies perturbations at
multiple sizes. The process is repeated five times (vectorized) and all unique
parameter values required for the sensitivity computation are collected.

Perturbations that create invalid values (<2) are ignored!

In [ ]:
# configuration
repeats = 5
num_segments = 10
min_sample_range = (2, 50)
deltas = np.array([[2], [5], [10]])

# sampled values
segments = np.linspace(*min_sample_range, num_segments + 1, dtype=int)
samples = np.random.uniform(segments[:-1], segments[1:], size=(repeats, num_segments))
min_sample_sizes = np.round(samples).astype(int).T

# perturbed values
directions = np.random.choice((-1, 1), size=(deltas.shape[0], num_segments, repeats))
perturbations = directions * deltas[:, :, np.newaxis]
new_values = min_sample_sizes[np.newaxis, :, :] + perturbations

# all values to evaluate
all_values = np.concatenate((min_sample_sizes.flatten(), new_values.flatten()))
all_values = np.unique(all_values)
all_values = all_values[(all_values >= 2)]

np.save("data/generated/benchmark_sensitivity_deltas.npy", deltas)
np.save("data/generated/benchmark_sensitivity_min_sizes.npy", min_sample_sizes)
np.save("data/generated/benchmark_sensitivity_perturbed_values.npy", new_values)

## 2. Compute quality measures

This stage computes clustering quality measures at the sampled parameter values.

First, we configure the datasets. This step assumes all dataset have been
downloaded and pre-processed before running this notebook. See instructions at
`docs/data/[data-set]/README.md`!

In [ ]:
def load_iris():
    X, y = sk_load_iris(return_X_y=True)
    return X, y.astype(np.intp)


def load_mnist():
    X, y = fetch_openml("mnist_784", version=1, return_X_y=True)
    return X.to_numpy(dtype=np.float32), y.cat.codes.to_numpy().astype(np.intp)


def load_fashion_mnist():
    X, y = fetch_openml("Fashion-MNIST", version=1, return_X_y=True)
    return X.to_numpy(dtype=np.float32), y.cat.codes.to_numpy().astype(np.intp)


def local_data_loader(folder):
    """Load data from a specified folder in the data directory.

    See `data/[data-set]/README.md` for processing details.
    """
    X_path = f"data/{folder}/generated/X.npy"
    y_path = f"data/{folder}/generated/y.npy"
    if not (os.path.exists(X_path) and os.path.exists(y_path)):
        raise FileNotFoundError(f"Data not found in folder: {folder}")

    def load_data():
        X = np.load(X_path)
        y = np.load(y_path)
        return X, y.astype(np.intp)

    return load_data


def as_float32(X):
    return X.astype(np.float32, copy=False)


def standardize_features(X):
    return StandardScaler().fit_transform(as_float32(X))


def l2_normalize_features(X):
    return normalize(as_float32(X), norm="l2")


def scale_pixels_to_unit_interval(X):
    X = as_float32(X)
    max_value = np.nanmax(X)
    if max_value > 1.0:
        X = X / 255.0
    return X


def pca_project(X, n_components=50):
    n_components = min(n_components, X.shape[0], X.shape[1])
    return PCA(
        n_components=n_components, svd_solver="randomized", random_state=0
    ).fit_transform(X)


def pixel_pca_preprocess(X):
    return pca_project(scale_pixels_to_unit_interval(X))


def standardized_pca_preprocess(X):
    return pca_project(standardize_features(X))


def normalized_pca_preprocess(X):
    return pca_project(l2_normalize_features(X))


preprocessing_groups = dict(
    dense_features_small=dict(
        datasets=[
            "iris",
            "authorship",
            "cardiotocography",
            "cell_cycle_237",
            "ecoli",
            "mfeat_karhunen",
            "yeast_galactose",
        ],
        preprocess=standardize_features,
        metric="euclidean",
        preprocessing="z-score",
    ),
    dense_features_large=dict(
        datasets=["mfeat_factors"],
        preprocess=standardized_pca_preprocess,
        metric="euclidean",
        preprocessing="z-score + PCA(50)",
    ),
    images_large=dict(
        datasets=["mnist", "fashion_mnist", "semeion"],
        preprocess=pixel_pca_preprocess,
        metric="euclidean",
        preprocessing="scale-to-[0,1] + PCA(50)",
    ),
    text_small=dict(
        datasets=["articles_1442_5", "articles_1442_80"],
        preprocess=l2_normalize_features,
        metric="cosine",
        preprocessing="L2-normalize",
    ),
    embeddings_large=dict(
        datasets=["audioset", "cifar_10", "newsgroups"],
        preprocess=normalized_pca_preprocess,
        metric="cosine",
        preprocessing="L2-normalize + PCA(50)",
    ),
    already_preprocessed=dict(
        datasets=["elegans"],
        preprocess=as_float32,
        metric="euclidean",
        preprocessing="used as loaded",
    ),
)

dataset_settings = {
    dataset_name: dict(
        preprocess=group_config["preprocess"],
        metric=group_config["metric"],
        preprocessing=group_config["preprocessing"],
    )
    for group_config in preprocessing_groups.values()
    for dataset_name in group_config["datasets"]
}

data_configs = dict(
    iris=dict(loader=load_iris, **dataset_settings["iris"]),
    mnist=dict(loader=load_mnist, **dataset_settings["mnist"]),
    fashion_mnist=dict(loader=load_fashion_mnist, **dataset_settings["fashion_mnist"]),
)

for folder in os.listdir("data"):
    if folder in ["generated", "clusterable"]:
        continue
    if not os.path.isdir(os.path.join("data", folder)):
        continue
    if folder not in dataset_settings:
        warnings.warn(
            f"Skipping {folder}. No preprocessing strategy configured for this benchmark."
        )
        continue

    try:
        loader = local_data_loader(folder)
        data_configs[folder] = dict(loader=loader, **dataset_settings[folder])
    except FileNotFoundError:
        warnings.warn(
            f"Skipping {folder}. See "
            f"`data/{folder}/README.md` for download instructions."
        )

Next, we implement the clustering quality measures:

In [ ]:
def compute_quality(true_labels, predicted_labels):
    non_noise = predicted_labels != -1

    ari = adjusted_rand_score(true_labels[non_noise], predicted_labels[non_noise])
    homogeneity = homogeneity_score(true_labels[non_noise], predicted_labels[non_noise])
    completeness = completeness_score(
        true_labels[non_noise], predicted_labels[non_noise]
    )
    if homogeneity + completeness == 0.0:
        v_measure = 0.0
    else:
        v_measure = 2.0 * homogeneity * completeness / (homogeneity + completeness)
    noise_fraction = 1 - non_noise.sum() / len(predicted_labels)
    return ari, homogeneity, completeness, v_measure, noise_fraction

Then, we create functions that evaluate the algorithms. These functions return
one or more records identifying the dataset, algorithm configuration, and
resulting quality scores. For PLSCAN, we compute scores for its top-$n$
layers:

In [ ]:
def evaluate_hdbscan(
    X, y, data_name, k, alg_name, params, post_params, metric, preprocessing
 ):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        start = time.perf_counter()
        labels = HDBSCAN(
            min_samples=k, min_cluster_size=k, metric=metric, **params
        ).fit_predict(X)
        duration = time.perf_counter() - start
    ari, homogeneity, completeness, v_measure, noise_fraction = compute_quality(
        y, labels
    )
    return [
        dict(
            ari=ari,
            homogeneity=homogeneity,
            completeness=completeness,
            v_measure=v_measure,
            noise_fraction=noise_fraction,
            data_set=data_name,
            preprocessing=preprocessing,
            distance_metric=metric,
            compute_time=duration,
            k=k,
            alg_id="_".join([alg_name] + [f"{v}" for v in params.values()]),
            alg_config={
                "algorithm": alg_name,
                "metric": metric,
                "preprocessing": preprocessing,
                **params,
            },
        )
    ]

In [ ]:
def evaluate_plscan(
    X, y, data_name, k, alg_name, params, post_params, metric, preprocessing
):
    # Compute PLSCAN layers
    start = time.perf_counter()
    c = PLSCAN(min_samples=k, min_cluster_size=k, metric=metric, **params).fit(X)
    duration = time.perf_counter() - start
    layers = c.cluster_layers(**post_params) or [(k, c.labels_, c.probabilities_)]

    # Rank layers by their persistence score
    sizes, pers = c._persistence_trace
    sizes = sizes if sizes.shape[0] > 0 else np.array([k], dtype=np.float32)
    pers = pers if pers.shape[0] > 0 else np.array([0.0], dtype=np.float32)
    s = [s for s, _, _ in layers]
    ranks = rankdata(-pers[np.searchsorted(sizes, s)], method="ordinal")
    records = []

    # Create a record and compute quality for each layer
    for peak, (size, labels, _) in zip(ranks, layers):
        ari, homogeneity, completeness, v_measure, noise_fraction = compute_quality(
            y, labels
        )
        records.append(
            dict(
                ari=ari,
                homogeneity=homogeneity,
                completeness=completeness,
                v_measure=v_measure,
                noise_fraction=noise_fraction,
                preprocessing=preprocessing,
                distance_metric=metric,
                compute_time=duration,
                data_set=data_name,
                k=k,
                alg_id="_".join(
                    [alg_name] + [f"{v}" for v in params.values()] + [f"{peak}"]
                ),
                alg_config={
                    "algorithm": alg_name,
                    "metric": metric,
                    "preprocessing": preprocessing,
                    **params,
                    "peak": peak,
                    "birth_size": size,
                },
            )
        )
    return records

Next, we specify the algorithm--parameter configurations. We want to evaluate
all parameter-value combinations. In this case, only the cluster selection
strategies vary.

In [ ]:
# Configurations
algorithms = dict(plscan=evaluate_plscan, hdbscan=evaluate_hdbscan)
in_params = defaultdict(
    dict, 
    plscan=dict(
        persistence_measure=[
            "size",
            "distance",
            "density",
            "size-distance",
            "size-density",
        ]
    ),
    hdbscan=dict(cluster_selection_method=["leaf", "eom"]),
)
post_params = defaultdict(dict, plscan=dict(max_peaks=[5]))
algorithm_configs = [
    (
        fun,
        name,
        {param: value for param, value in zip(in_params[name].keys(), param_values)},
        {
            param: value
            for param, value in zip(post_params[name].keys(), post_param_values)
        },
    )
    for name, fun in algorithms.items()
    for param_values in product(*in_params[name].values())
    for post_param_values in product(*post_params[name].values())
]

Finally, run all algorithm--dataset--$k$ combinations (takes about 3 hours). 

Each dataset is preprocessed once with a fixed rule chosen by data type and size.
Dense numeric datasets are standardized and clustered with Euclidean distances.
Semantic embedding and article-vector datasets are L2-normalized and clustered
with cosine distances. Large raw image datasets are scaled to $[0, 1]$ and
projected to 50 principal components before clustering with Euclidean
distances. This keeps the benchmark computationally reasonable without adding
another nonlinear representation-learning step.

In [ ]:
records = []

pbar = tqdm(total=len(data_configs) * len(all_values) * len(algorithm_configs))
for data_name, config in data_configs.items():
    # Load and preprocess the data once per dataset
    X, y = config["loader"]()
    X_preprocessed = config["preprocess"](X)
    metric = config["metric"]
    preprocessing = config["preprocessing"]
    np.save(
        f"data/generated/benchmark_sensitivity_{data_name}_processed.npy",
        X_preprocessed,
    )

    # Evaluate the algorithms
    for k, (evaluator, alg_name, params, post_params) in product(
        all_values, algorithm_configs
    ):
        if k < X_preprocessed.shape[0] - 1:
            records.extend(
                evaluator(
                    X_preprocessed,
                    y,
                    data_name,
                    k,
                    alg_name,
                    params,
                    post_params,
                    metric,
                    preprocessing,
                )
            )
        pbar.update()


df = pd.DataFrame.from_records(records)
df.to_parquet("data/generated/benchmark_sensitivity.parquet")
df.head()

## 2.5 Plot the results

This section plots the $k$--quality curves for all evaluated algorithms
configurations and datasets.

First, we load the data generated by the previous steps.

In [ ]:
df = pd.read_parquet("data/generated/benchmark_sensitivity.parquet")
deltas = np.load("data/generated/benchmark_sensitivity_deltas.npy")
min_sample_sizes = np.load("data/generated/benchmark_sensitivity_min_sizes.npy")
new_values = np.load("data/generated/benchmark_sensitivity_perturbed_values.npy")

Then, we configure display names for the variables and values.

In [ ]:
def to_display_name(input):
    parts = input.split("_")
    name = parts[0].upper()
    if name == "HDBSCAN":
        name = "HDBSCAN*"
        return f"{name} ({' '.join(parts[1:])})"
    if name == "PLSCAN":
        params = [
            p.replace("-", "–").replace("density", "λ").replace("distance", "d")
            for p in parts[1:]
        ]
        return f"{name} ({' '.join(params)})"
    return input


def dataset_name(input):
    names = dict(
        iris="Iris",
        mnist="MNIST",
        fashion_mnist="Fashion-MNIST",
        articles_1442_5="Articles-1442-5",
        articles_1442_80="Articles-1442-80",
        audioset="AudioSet (music)",
        authorship="Authorship",
        cardiotocography="CTG",
        cell_cycle_237="CellCycle-237",
        cifar_10="CIFAR-10",
        ecoli="E. coli",
        elegans="C. elegans",
        mfeat_factors="Mfeat-Factors",
        mfeat_karhunen="Mfeat-Karhunen",
        newsgroups="20 Newsgroups",
        semeion="Semeion Digits",
        yeast_galactose="YeastGalactose",
    )
    return names.get(input, input)

Next, we summarize PLSCAN's layers into the default (most-persistent) layer, and
the optimal layer at each $k$. The latter strategy mimics a workflow that
manually inspects and selects a cluster layer when tuning the algorithm.

In [ ]:
# Create new data frame without the rows for non-default plscan layers
def digit_higher_than_one(char):
    return char.isdigit() and int(char) > 1


mask = df.alg_id.apply(lambda x: not digit_higher_than_one(x.split("_")[-1]))
df_default = df[mask].copy()
df_default.alg_id = df_default.alg_id.apply(
    lambda x: x if "plscan" not in x else "_".join(x.split("_")[:-1])
)

In [ ]:
# Select rows with the best plscan layer at each dataset--k
plscan_df = df.query("alg_id.str.startswith('plscan')", engine="python").copy()
plscan_df["persistence"] = plscan_df.alg_id.apply(lambda x: x.split("_")[1])
plscan_df["layer"] = plscan_df.alg_id.apply(lambda x: x.split("_")[-1])
best_layer = (
    plscan_df.groupby(["data_set", "persistence", "k"])
    .apply(lambda g: g.loc[g.ari.idxmax()], include_groups=False)
    .reset_index()
)
best_layer.alg_id = best_layer.alg_id.apply(
    lambda x: "_".join(x.split("_")[:-1] + ["top"])
)
best_layer = best_layer.drop(columns=["layer", "persistence"])

Maximum observed V-measure per dataset and algorithm:

In [ ]:
# Combine these dataframes
df_top = pd.concat((df_default, best_layer), ignore_index=True)
df_top.groupby(["data_set", "alg_id"]).apply(
    lambda g: g.loc[g.v_measure.idxmax(), ["v_measure"]],
    include_groups=False,
).reset_index().round(3).pivot(
    index="alg_id", columns="data_set", values=["v_measure"]
)

ARI values at $k=4$ for the default layers:

In [ ]:
columns = [
    "hdbscan_eom",
    "hdbscan_leaf",
    "plscan_size",
    "plscan_size-distance",
    "plscan_size-density",
    "plscan_distance",
    "plscan_density",
]

tab = (
    df_default.query("k == 4")
    .pivot(index="data_set", columns="alg_id", values="ari")
    .round(2)
)

tab[columns]

Diagnostic companion table using average ranks at $k=4$ for the default layers.

Lower average rank is better. The score-based diagnostics rank larger values higher,
while the noise diagnostic ranks smaller values higher.

In [ ]:
rank_metrics = dict(
    v_measure=False,
    homogeneity=False,
    completeness=False,
    noise_fraction=True,
)

companion_df = df_default.query("k == 4").copy()
rank_columns = [
    "hdbscan_eom",
    "hdbscan_leaf",
    "plscan_size",
    "plscan_size-distance",
    "plscan_size-density",
    "plscan_distance",
    "plscan_density",
]

rank_table = pd.concat(
    [
        companion_df.pivot(index="data_set", columns="alg_id", values=metric)
        .rank(axis=1, method="average", ascending=ascending)
        .mean(axis=0)
        .rename(metric)
        for metric, ascending in rank_metrics.items()
    ],
    axis=1,
).loc[rank_columns]

rank_table = rank_table.rename(
    columns={
        "v_measure": "V-measure rank",
        "homogeneity": "Homogeneity rank",
        "completeness": "Completeness rank",
        "noise_fraction": "Noise rank",
    }
).round(2)

rank_table

Next, we configure the colors, titles, and plotting orders:

In [ ]:
# Configure plotting order, colors and titles
plot_df = df_top.copy()
data_sets = sorted(plot_df.data_set.unique())
alg_ids = [
    "hdbscan_eom",
    "hdbscan_leaf",
    "plscan_size_top",
    "plscan_size",
    "plscan_size-distance_top",
    "plscan_size-distance",
    "plscan_size-density_top",
    "plscan_size-density",
    "plscan_distance_top",
    "plscan_distance",
    "plscan_density_top",
    "plscan_density",
]
palette = [
    mpl.colors.to_rgb("C0"),
    mpl.colors.to_rgb("C1"),
    lighten("C2"),
    mpl.colors.to_rgb("C2"),
    lighten("C3"),
    mpl.colors.to_rgb("C3"),
    lighten("C4"),
    mpl.colors.to_rgb("C4"),
    lighten("C5"),
    mpl.colors.to_rgb("C5"),
    lighten("C6"),
    mpl.colors.to_rgb("C6"),
]
titles = [
    "HDBSCAN*\nEOM/leaf",
    "PLSCAN\nsize",
    "PLSCAN\nsize–d",
    "PLSCAN\nsize–λ",
    "PLSCAN\nd",
    "PLSCAN\nλ",
]

Now, we plot the ARI as a curve over $k$. We interested in the range of $k$
values for which the algorithms produce high quality clusterings.

In [ ]:
# Create the plots
plt.figure(figsize=(2.75 * 3, 0.6))
max_x = plot_df.k.max()
ticks = [0, 0.5, 1]
alg_idxs = [[0, 1], [2, 3], [4, 5], [6, 7], [8, 9], [10, 11]]
for j, ids in enumerate(alg_idxs):
    plt.subplot(1, 6, j + 1)
    plt.title(titles[j], y=0)
    plt.xlim(0, max_x)
    plt.xticks([0, 25, 50])
    plt.yticks([])
    plt.ylim(0, 1)
    if j == 0:
        plt.ylabel(f"{dataset_name(data_sets[0])}\nARI.", labelpad=0, color="w")
    plt.tick_params(axis="x", colors="white")
    plt.tick_params(axis="y", colors="white")
    plt.suptitle("Per dataset curves", y=1)
    plt.subplots_adjust(bottom=0, right=1, top=1, left=0.08, wspace=0.01)
    frame_off()

In [ ]:
for i, data_name in enumerate(data_sets):
    plt.figure(figsize=(2.75 * 3, 1.75))
    ddf = plot_df.query(f'data_set == "{data_name}"')
    max_x = ddf.k.max()
    max_y = ddf.ari.max()
    for j, ids in enumerate(alg_idxs):
        plt.subplot(1, 6, j + 1)
        # frame_off()
        plt.hlines(
            max_y,
            0,
            max_x,
            colors=lighten("k"),
            linestyles=":",
            linewidth=0.5,
        )
        sns.lineplot(
            data=ddf,
            x="k",
            y="ari",
            hue="alg_id",
            hue_order=[alg_ids[idx] for idx in ids],
            errorbar=None,
            palette=[palette[idx] for idx in ids],
            legend=False,
        )
        sns.lineplot(
            data=ddf,
            x="k",
            y="noise_fraction",
            hue="alg_id",
            hue_order=[alg_ids[idx] for idx in ids],
            errorbar=None,
            palette=[palette[idx] for idx in ids],
            linestyle=":",
            linewidth=1,
            legend=False,
        )
        plt.xlabel("$k$", labelpad=0)
        plt.xticks([0, 25, 50])
        plt.ylim(0, 1.05)
        plt.yticks(ticks)
        if j == 0:
            plt.ylabel(f"{dataset_name(data_name)}\nARI", labelpad=0)
        else:
            plt.ylabel("")
            plt.gca().set_yticklabels(["" for t in ticks])
        plt.subplots_adjust(bottom=0.26, right=1, left=0.08, wspace=0.01, top=0.98)
plt.show()

Next, we summarize the shape of the curves over all datasets by Lowess
interpolation. We scaled the ARI scores by the maximum value achieved on each
dataset to compare the curves, rather than the exact values. Still, this
interpolation has issues because some datasets need other $k$ values to get good
clusters.

In [ ]:
def scale_quality(group):
    max_q = group.ari.max()
    group["scaled_quality"] = group.ari / max_q
    return group


plot_df = (
    plot_df.groupby(["data_set"])
    .apply(scale_quality, include_groups=False)
    .reset_index()
)

In [ ]:
plt.figure(figsize=(2.75 * 3, 2))
max_y = plot_df.scaled_quality.max()
for j, ids in enumerate(alg_idxs):
    plt.subplot(1, 6, j + 1)
    for i, idx in enumerate(ids):
        alg_id = alg_ids[idx]
        plt.hlines(max_y, 0, max_x, colors="k", linestyles=":", linewidth=0.5)
        regplot_lowess_ci(
            plot_df.query(f'alg_id == "{alg_id}"'),
            x="k",
            y="scaled_quality",
            ci_level=95,
            n_boot=100,
            lowess_frac=0.05,
            color=palette[idx],
            scatter=False,
        )
        plt.title(titles[j])
        plt.xlabel("k", labelpad=0)
        plt.ylim(0, 1)
        plt.yticks([])
        plt.xticks([0, 25, 50])
        if j == 0:
            plt.ylabel("Scaled ARI", labelpad=0)
        else:
            plt.ylabel("")
plt.suptitle("Interpolated ARI curves", y=1)
plt.subplots_adjust(bottom=0.22, right=1, left=0.02, wspace=0.02, top=0.68)
plt.show()

## 3. Compute sensitivity

Now, we actually compute and compare the sensitivity measure. The measure
describes how much  quality scores change when the $k$ parameter changes. So, it
already takes into account absolute quality differences between datasets.

In [ ]:
sensitivity_records = []
for data_set, alg_id in product(df_top.data_set.unique(), df_top.alg_id.unique()):
    sub_df = df_top.query(f"data_set == '{data_set}' & alg_id == '{alg_id}'")
    vals = defaultdict(lambda: np.nan, {k: v for k, v in zip(sub_df.k, sub_df.ari)})
    lookup_fun = np.vectorize(lambda x: vals[x])
    initial_val = lookup_fun(min_sample_sizes)[np.newaxis, :, :]
    perturbed_val = lookup_fun(new_values)
    with np.errstate(divide="ignore", invalid="ignore"):
        diff = np.abs((perturbed_val - initial_val) / (perturbed_val + initial_val))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        sensitivity = np.nanmean(diff, axis=(1, 2))
    for delta, sensitivity in zip(deltas[:, 0], sensitivity):
        sensitivity_records.append(
            {
                "data_set": data_set,
                "alg_id": alg_id,
                "perturbation": delta,
                "sensitivity": sensitivity,
            }
        )

# Convert to pandas
df_sens = pd.DataFrame.from_records(sensitivity_records)
df_sens.head()

On these datasets, PLSCAN has a lower sensitivity to $k$ than HDBSCAN*. However,
Peng et al. classify all values below 0.25 as insensitive. 

In [ ]:
plt.figure(figsize=(2.8 * 3, 2))

plot_df = df_sens.copy()
plot_df.alg_id = plot_df.alg_id.apply(to_display_name)

alg_order = [0, 1, 3, 2, 5, 4, 7, 6, 9, 8, 11, 10]
hue_order = [to_display_name(alg_ids[x]) for x in alg_order]
alg_palette = [palette[x] for x in alg_order]
ax = sns.violinplot(
    data=plot_df,
    y=plot_df.sensitivity,
    x=pd.Categorical(plot_df.perturbation),
    hue=plot_df.alg_id,
    hue_order=hue_order,
    density_norm="count",
    palette=alg_palette,
    legend=True,
    inner="quartiles",
    linewidth=0.5,
    cut=0,
)
max_y = 0.3
plt.ylim(-0.005, max_y)
plt.legend(ncol=4, title="", loc="upper left")
plt.xlabel("Perturbation of $k$")
plt.ylabel("Avg.~sensitivity")
plt.title("Parameter sensitivity distributions")
plt.subplots_adjust(left=0.075, right=1, bottom=0.21, top=0.9)
plt.show()

## 4. Explore specific datasets

In [ ]:
X = np.load("data/generated/benchmark_sensitivity_yeast_galactose_processed.npy")
c = PLSCAN().fit(X)
c.leaf_tree_.plot()
plt.show()

In [ ]:
df.data_set.unique()

In [ ]:
y = np.load("data/ecoli/generated/y.npy")
u, c = np.unique(y, return_counts=True)
c

In [ ]:
X = np.load("data/generated/benchmark_sensitivity_ecoli_processed.npy")
c = PLSCAN().fit(X)
c.leaf_tree_.plot()
plt.show()